<a href="https://colab.research.google.com/github/YOUR-USERNAME/bags-vectors-transformers/blob/main/day3/notebooks/1_bert_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Day 3 — Contextual Embeddings with BERT

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the notebook everything has been building toward. We finally use a real **transformer**
(BERT) and see the thing static embeddings could not do: give a word a **different vector
depending on its sentence**.

By the end you will be able to:

- Run a state-of-the-art model in **three lines** with the `pipeline` helper
- See how BERT **tokenizes** text into subwords, with `[CLS]` / `[SEP]` and the 512-token limit
- Do the **"bank" test**: prove contextual vectors change with context
- Use a model **off-the-shelf** to classify text
- **Explore parameters** (model size, truncation) and their effects
- *(Optional)* **Fine-tune** a model on the policy-bills task

> **Important — turn on the GPU!** In Colab: *Runtime → Change runtime type → T4 GPU*.
> Transformers are slow on CPU. Several cells download models the first time (be patient).


## 0. Setup

We install the Hugging Face **`transformers`** library (the models) and **`datasets`** (for
the policy data later). Colab already has PyTorch and, if you enabled it, a GPU.


In [ ]:
!pip install transformers datasets -q

import torch
from transformers import pipeline, AutoTokenizer, AutoModel

# Check whether a GPU is available — this makes everything much faster
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device.upper())
if device == "cpu":
    print("⚠️  No GPU detected. Things will be slow. "
          "Consider: Runtime → Change runtime type → T4 GPU.")

## 1. The easiest possible start: `pipeline`

Hugging Face's `pipeline` helper hides all the complexity. You name a **task**, and it
downloads a suitable pre-trained model and runs it. Let's do sentiment analysis.


In [ ]:
# Create a ready-made sentiment classifier (downloads a model the first time)
sentiment = pipeline("sentiment-analysis")

# Run it on some text
result = sentiment("This new policy is an absolute disaster.")
print(result)

That's it — a real, state-of-the-art model, running on your text, in three lines. The
output gives a **label** (POSITIVE/NEGATIVE) and a **confidence score**.

Let's try a few examples at once, including a tricky one.


In [ ]:
examples = [
    "I love the new public transport plan!",
    "This is the worst decision the city has ever made.",
    "The meeting is scheduled for Tuesday afternoon.",  # neutral-ish
    "Well, that went just great.",                       # sarcasm — hard!
]

for text in examples:
    r = sentiment(text)[0]
    print(f"{r['label']:<9} ({r['score']:.2f})  {text}")

> **✏️ Exercise 1**
>
> Write three sentences of your own — ideally some ambiguous or sarcastic ones — and run them
> through `sentiment`. Where does it do well? Where does it stumble? (Even powerful models
> struggle with sarcasm and context-dependent meaning.)


In [ ]:
# Your code here


## 2. Under the hood: tokenization

The `pipeline` did a lot silently. The first step it hides is **tokenization** — turning your
text into the numbers the model actually reads. Let's look at it directly, using the
tokenizer from a small, fast model (**DistilBERT**).


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

text = "Transformers are powerful."
tokens = tokenizer.tokenize(text)
print("Text:  ", text)
print("Tokens:", tokens)

Notice BERT works with **subword tokens**, not whole words. Common words stay whole, but
rarer or longer words get split into pieces (marked with `##`). This is how BERT handles words
it has never seen — the same idea as fastText from Day 2.


In [ ]:
# A made-up / rare word gets split into subword pieces
for word in ["preprocessing", "unbelievable", "gezondheidszorg", "antidisestablishmentarianism"]:
    print(f"{word:<32} -> {tokenizer.tokenize(word)}")

### Special tokens and IDs

When BERT actually encodes text, it adds two **special tokens**:
- `[CLS]` at the start — a slot that summarizes the whole sequence (used for classification)
- `[SEP]` at the end — marks the end of a sentence

And every token becomes an integer **ID** (its index in the vocabulary).


In [ ]:
encoded = tokenizer("Transformers are powerful.")
ids = encoded["input_ids"]

print("Token IDs:  ", ids)
print("Back to tokens:", tokenizer.convert_ids_to_tokens(ids))
print()
print("Notice [CLS] at the start and [SEP] at the end.")

### The 512-token limit

From the lecture: BERT can only read **512 tokens at once**. Longer documents must be split or
truncated. Let's confirm the limit and see truncation in action.


In [ ]:
print("Max input length for this model:", tokenizer.model_max_length, "tokens")

# A very long text, truncated to a max length
long_text = "policy " * 20   # 20 repeats
enc = tokenizer(long_text, max_length=10, truncation=True)
print("\nWith max_length=10, truncation=True:")
print("Number of tokens kept:", len(enc["input_ids"]))
print("Tokens:", tokenizer.convert_ids_to_tokens(enc["input_ids"]))

> **✏️ Exercise 2**
>
> Tokenize a sentence in **Dutch** (or another language) with this English tokenizer, e.g.
> `"De nieuwe wet is een ramp"`. How does it split the words? What does that tell you about
> why you'd want a **Dutch** tokenizer / model (like BERTje) for Dutch text?


In [ ]:
# Your code here


## 3. The payoff: the "bank" test

This is the heart of the whole notebook. On Day 2, static embeddings gave "bank" **one fixed
vector** — identical in "river bank" and "money bank". BERT gives a **different** vector for
each, because it reads the whole sentence.

We load the model itself (not just the tokenizer) and extract the vector for "bank" in two
different sentences.


In [ ]:
model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
model.eval()   # evaluation mode (no training)
print("Model loaded.")

In [ ]:
def get_word_vector(sentence, target_word):
    """Return BERT's contextual vector for target_word in a given sentence."""
    # Tokenize, keeping track of token positions
    enc = tokenizer(sentence, return_tensors="pt").to(device)
    with torch.no_grad():                      # no gradients needed — we're not training
        output = model(**enc)
    # output.last_hidden_state: one vector per token
    hidden = output.last_hidden_state[0]       # [num_tokens, hidden_size]

    # Find which token(s) correspond to our target word
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    target_tokens = tokenizer.tokenize(target_word)
    # Find the position of the first target sub-token
    for i, t in enumerate(tokens):
        if t == target_tokens[0]:
            return hidden[i].cpu().numpy()
    raise ValueError(f"'{target_word}' not found in tokens: {tokens}")

# "bank" in two very different senses
s1 = "I sat on the grassy bank of the river."
s2 = "I deposited my paycheck at the bank."

v1 = get_word_vector(s1, "bank")
v2 = get_word_vector(s2, "bank")

print("Got two vectors for 'bank', each of size", v1.shape[0])

Now the crucial comparison. If BERT is truly contextual, these two "bank" vectors should
be **different**. Let's measure their cosine similarity — and compare against the *same* word
used in the *same* sense twice.


In [ ]:
import numpy as np

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Same word, DIFFERENT senses
diff_sense = cosine(v1, v2)

# Same word, SAME sense (two river sentences) — should be more similar
s3 = "We fished from the bank of the stream."
v3 = get_word_vector(s3, "bank")
same_sense = cosine(v1, v3)

print(f"'bank' river  vs  'bank' money  (different sense): {diff_sense:.3f}")
print(f"'bank' river  vs  'bank' stream (same sense):      {same_sense:.3f}")
print()
print("The same-sense pair is more similar — BERT encodes the CONTEXT, not just the word!")

**This is the whole point of contextual embeddings.** The string "bank" is identical in
all three sentences, but BERT gives it vectors that reflect its **meaning in context**. A
static embedding (Word2Vec, GloVe) would give the exact same vector every time — similarity
1.000 across the board. BERT can tell the river from the money.

> **✏️ Exercise 3**
>
> Try the same test with another ambiguous word: **"spring"** (season / coil / to jump), or
> **"left"** (direction / departed / political). Write three sentences using different senses
> and compare the vectors. Does BERT separate the senses?


In [ ]:
# Your code here


## 4. Back to policy: zero-shot classification

Let's reconnect to the **policy bills** data. Suppose we want to sort bill titles into policy
areas but have **no labeled training data** of our own. A neat trick: **zero-shot
classification**. You give the model candidate labels in plain English, and it picks the best
fit — no training at all.


In [ ]:
# Zero-shot classifier: you supply the candidate labels at runtime
zero_shot = pipeline("zero-shot-classification")

bill_title = "A bill to expand access to affordable health insurance coverage."
candidate_labels = ["health", "education", "taxation", "defense", "environment"]

result = zero_shot(bill_title, candidate_labels)
print("Title:", bill_title, "\n")
for label, score in zip(result["labels"], result["scores"]):
    print(f"  {label:<12} {score:.3f}")

The model never saw "health" as a training label — it reasons about the *meaning* of the
title and the labels. The top score should be "health". This is remarkably useful when you
have categories in mind but no labeled corpus.

> **✏️ Exercise 4**
>
> Run zero-shot classification on two or three bill titles of your own invention, using the
> same candidate labels. Does it pick sensible policy areas? Try a title that could plausibly
> fit **two** areas and see how it splits the scores.


In [ ]:
# Your code here


## 5. Exploring parameters: model size and speed

Not all models are equal. A key practical trade-off is **size vs. speed**. Let's compare a
**small** model (DistilBERT) with a **larger** one on the same task, and time them.


In [ ]:
import time

text = "The government announced a major new economic reform package today."

# Small, fast model
small = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

start = time.time()
for _ in range(20):
    small(text)
small_time = time.time() - start
print(f"DistilBERT (small): {small_time:.2f}s for 20 runs")

DistilBERT is about **40% smaller and 60% faster** than full BERT, with only a small
accuracy cost. For prototyping and large corpora, that trade is usually worth it. Start small,
scale up only if you need the accuracy.

### Tokenization parameters that matter

When you feed text to a model, a few parameters control how it's prepared. These matter a lot
in practice:


In [ ]:
sample = "This is a fairly long sentence that we will use to demonstrate truncation and padding."

# max_length + truncation: cap the number of tokens
enc_trunc = tokenizer(sample, max_length=8, truncation=True)
print("Truncated to 8 tokens:", tokenizer.convert_ids_to_tokens(enc_trunc["input_ids"]))
print()

# padding: pad short texts to a fixed length so they can be batched together
batch = tokenizer(["short text", "a somewhat longer piece of text here"],
                  padding=True, truncation=True)
for ids in batch["input_ids"]:
    print(f"length {len(ids)}:", tokenizer.convert_ids_to_tokens(ids))

- **`max_length` + `truncation`**: essential for long documents (respects the 512 limit)
- **`padding`**: makes all sequences the same length so you can process them in **batches** —
  important for speed when you have many documents

> **✏️ Exercise 5**
>
> Take a long bill title (or make one up) and tokenize it three times with `max_length` set to
> 5, 15, and 30. Print how many tokens survive each time. How would truncating too aggressively
> hurt a classifier?


In [ ]:
# Your code here


## 6. *(Optional)* Fine-tuning on the policy data

Everything so far used models **off-the-shelf**. The most powerful option is **fine-tuning**:
updating the model on *your* labeled task. This is heavier — it **really wants a GPU** and takes
a few minutes — so treat this section as optional.

We fine-tune DistilBERT to classify bill titles into policy areas, using a small subset for
speed.

> ⚠️ **This section needs a GPU** (*Runtime → Change runtime type → T4 GPU*) and will be slow
> otherwise. If you're short on time, just read along.


In [ ]:
from datasets import load_dataset

# Load the policy bills, keep a few common areas, small balanced sample for speed
bills = load_dataset("dreamproit/bill_labels_us", split="train").to_pandas()
bills = bills.rename(columns={"title": "text"})[["text", "policy_area"]].dropna()

top_areas = bills["policy_area"].value_counts().head(4).index.tolist()
bills = bills[bills["policy_area"].isin(top_areas)]
bills = bills.groupby("policy_area", group_keys=False).apply(
    lambda g: g.sample(min(len(g), 400), random_state=42)
).reset_index(drop=True)

# Map string labels to integers (models need integer labels)
labels = sorted(bills["policy_area"].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
bills["label"] = bills["policy_area"].map(label2id)

print("Fine-tuning on", len(bills), "bills across", len(labels), "policy areas:")
print(labels)

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)

# Train/test split
train_df, test_df = train_test_split(bills, test_size=0.2, random_state=42,
                                     stratify=bills["label"])

# Convert to HuggingFace Datasets and tokenize
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=32)

train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_df).map(tokenize, batched=True)

# Load a classification model with the right number of labels
clf_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)
print("Ready to fine-tune.")

In [ ]:
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

# Training configuration — these are the knobs from the lecture
args = TrainingArguments(
    output_dir="./bert-policy",
    num_train_epochs=2,              # 2-4 is usually enough when fine-tuning
    per_device_train_batch_size=16,  # batch size
    learning_rate=2e-5,              # the standard fine-tuning learning rate
    eval_strategy="epoch",
    logging_steps=20,
    report_to="none",
)

trainer = Trainer(
    model=clf_model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# Evaluate the fine-tuned model
metrics = trainer.evaluate()
print(f"Fine-tuned accuracy: {metrics['eval_accuracy']:.3f}")

In [ ]:
# Try it on a new title
fine_tuned = pipeline("text-classification", model=clf_model, tokenizer=tokenizer)
print(fine_tuned("A bill to increase funding for public schools and teachers."))

> **✏️ Exercise 6** *(optional, if you ran the fine-tuning)*
>
> Change `num_train_epochs` to 1 and then to 4, and `learning_rate` to `5e-5`. How does
> accuracy change? Fine-tuning is sensitive to these choices — there's no universally best
> setting, which is why you always validate on held-out data.


In [ ]:
# Your code here (optional)


## Wrap-up

You have now worked with a real transformer, end to end:

- Ran a state-of-the-art model in **three lines** with `pipeline`
- Saw BERT's **subword tokenization**, special tokens, and the 512-token limit
- Proved the **"bank" test**: contextual vectors change with context (unlike static embeddings)
- Used **zero-shot classification** to sort policy text with no training data
- **Explored parameters**: model size/speed, truncation, padding
- *(Optional)* **Fine-tuned** a model on the policy-bills task

This completes the journey: from **bags** of words, to static **vectors**, to context-aware
**transformers**. You now have the full modern toolkit for computational text analysis — and,
just as importantly, a feel for **when each approach is the right one**.

### Where to go next

- Browse the **Hugging Face Hub** for models in your domain or language (search "Dutch",
  "BERTje", "policy", etc.)
- For your own research: start with the **lightest** tool that works (a dictionary or TF-IDF
  baseline), and reach for transformers when meaning and context genuinely matter.
- Always keep the **Day 1 discipline**: a held-out test set, and an honest baseline to beat.
